# QC0426 / VTI — Measurement

**QC0426** (trilayer graphene, stacking under investigation) on the
**VTI** rig, 2026-06-01 cooldown. Live HERA test session **2026-06-10**:
Qingrui at the keyboard runs every cell; the agent is advisory via nb-collab.

`metadata.hera.policy = append_only` — scientific record: append new cells;
don't edit or delete executed ones.

Instrument addresses and gate ranges are from Qingrui's
`20260601-QC0426-3LGr/measurement.ipynb` — confirm each `# <-- verify` line at
bring-up. Cells marked **PATTERN** / **PROPOSAL** contain `<PLACEHOLDER>`
values or pending decisions — review before running.

## Setup

In [1]:
import os
import time

import numpy as np
import scipy.constants as cst

import qcodes
from qcodes import validators
from qcodes.instrument_drivers.stanford_research import SR830
from qcodes.instrument_drivers.tektronix.Keithley_2400 import Keithley_2400

import sweep

from vti_drivers.FastBi1K5 import FastBi1K5
from vti_drivers.Lakeshore_335 import Lakeshore_335
from vti_drivers.SRS_DC205 import SRS_DC205
from vti_drivers.async_station import async_station

In [2]:
main_dir = os.getcwd()
sweep.set_basedir(os.path.join(main_dir, "data"))
main_dir

'C:\\Users\\DGG\\VTI_local_data\\zack\\20260610-QC0426-hera-live'

# Connect Instruments

## Magnet

In [3]:
magnet = FastBi1K5('magnet', '192.168.0.10')  # <-- verify address
magnet.status()

Connected to: Fast-Bi-1K5 Magnet Controller
---------- Magnet Status ----------
Field:               : 0.002 Tesla
Current:             : 0.015 A
Voltage:             : -0.000 V
Output Enabled       : True
Persistent Switch On : True
Loop Mode            : CC
Update Mode          : NORMAL
-----------------------------------


## Lakeshore

In [4]:
heater = Lakeshore_335('l335_3', 'GPIB0::10::INSTR')  # <-- verify address
print(heater.sample_temperature())
print(heater.vaporizer_temperature())

Connected to: LSCI 335 (serial:LSA2FM7/#######, firmware:1.4) in 0.47s
1.157
0.0


## Lock-ins

In [5]:
sr0 = SR830('sr0', "GPIB0::17::INSTR")  # <-- verify — current channel (Ithaco preamp output)
sr1 = SR830('sr1', "GPIB0::14::INSTR")  # <-- verify — also the AC drive source
sr2 = SR830('sr2', "GPIB0::19::INSTR")  # <-- verify
sr3 = SR830('sr3', "GPIB0::18::INSTR")  # <-- verify

C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch1_databuffer on instrument sr0 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch2_databuffer on instrument sr0 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch1_databuffer on instrument sr1 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.p

Connected to: Stanford_Research_Systems SR830 (serial:s/n45620, firmware:ver1.07) in 0.08s
Connected to: Stanford_Research_Systems SR830 (serial:s/n83412, firmware:ver1.07) in 0.08s
Connected to: Stanford_Research_Systems SR830 (serial:s/n41639, firmware:ver1.07) in 0.10s
Connected to: Stanford_Research_Systems SR830 (serial:s/n93552, firmware:ver1.07) in 0.08s


C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch1_databuffer on instrument sr3 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch2_databuffer on instrument sr3 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(


## Keithleys

In [6]:
k_bg = Keithley_2400('k_bg', 'GPIB0::24::INSTR')  # <-- verify address — back gate
k_bg.mode.set('VOLT')
k_bg.volt.set(0)
k_bg.rangei.set(1e-6)
k_bg.rangev.set(21)
k_bg.volt.step = 0.005

Connected to: KEITHLEY INSTRUMENTS INC. 2400 (serial:0734554, firmware:C17   Jul 16 1999 15:20:40/A02  /G/F) in 0.02s


In [7]:
k_2410 = Keithley_2400('k_2410', 'GPIB0::25::INSTR')  # <-- verify address — Si global gate
k_2410.mode.set('VOLT')
k_2410.volt.set(0)
k_2410.rangei.set(1e-6)
k_2410.rangev.set(21)
k_2410.volt.step = 0.005

Connected to: KEITHLEY INSTRUMENTS INC. 2410 (serial:1293333, firmware:C30   Mar 17 2006 09:29:29/A02  /J/J) in 0.03s


## DC205 (top gate)

In [8]:
k_tg = SRS_DC205("k_tg", "ASRL7::INSTR")  # <-- verify address — top gate
k_tg.ramp_rate = 2  # V/s

Connected to: Stanford_Research_Systems DC205 (serial:s/n20500514, firmware:ver1.80) in 0.02s


In [9]:
# Sweepable top-gate parameter — appears in the data as column 'DC_bias'.
def dcbias(vg):
    k_tg.ramp_voltage(vg)

dc205_volt = qcodes.Parameter('DC_bias', unit='V', set_cmd=dcbias)

# SAFETIES

Run immediately after instrument bring-up, before any sweep.

In [10]:
# Gate validators — QC0426, 2026-06-01 cooldown (ranges established in cells
# 50/62 of Qingrui's measurement notebook).
TG_MIN, TG_MAX = -6.5, +7.5   # top gate (dc205_volt / k_tg), V
BG_MIN, BG_MAX = -5.0, +4.8   # back gate (k_bg), V
SI_MIN, SI_MAX = -80.0, +100.0  # Si gate (k_2410), V  # [VERIFY with Qingrui] — he stepped -50..+100 V in Si maps; session plan stays at +50/+95
dc205_volt.vals = validators.Numbers(TG_MIN, TG_MAX)
k_bg.volt.vals = validators.Numbers(BG_MIN, BG_MAX)
k_2410.volt.vals = validators.Numbers(SI_MIN, SI_MAX)

# Limit lists for sanitize_tg_bg()
tg_lims, bg_lims = [TG_MIN, TG_MAX], [BG_MIN, BG_MAX]

# Session field cap — protocol-enforced (FastBi1K5 has no soft-limit hook).
# Every field vector / magnet.mag() target must satisfy |B| <= B_SESSION_CAP.
B_SESSION_CAP = 8.0  # Tesla


# Helpers

In [11]:
def spoints(a, b, c):
    return np.linspace(a, b, c)


def sanitize_tg_bg(vtg_vec, vbg_vec, tg_lims, bg_lims):
    """
    Given a list of tg and bg voltages, removes any points which are outside tg_lims and bg_lims
    """
    vtg_new_vec = []
    vbg_new_vec = []
    for ind in range(len(vtg_vec)):
        vtg, vbg = vtg_vec[ind], vbg_vec[ind]
        if vtg < tg_lims[0] or vtg > tg_lims[1] or vbg < bg_lims[0] or vbg > bg_lims[1]:
            pass
        else:
            vtg_new_vec.append(vtg)
            vbg_new_vec.append(vbg)
    return np.array(vtg_new_vec), np.array(vbg_new_vec)

In [12]:
cst_epsilon0 = cst.epsilon_0
cst_e = cst.e
cst_h = cst.h
cst_phi0 = cst_h/(2*cst_e)
cst_kappa_SiO2 = 3.9
cst_kappa_hBN = 3

def calc_ctg(t_tg):
    """
    Calculates area capacitance for a top gate
    """
    return cst_epsilon0*cst_kappa_hBN/t_tg

def calc_cbg(t_bg, t_bgSiO2):
    """
    Calculates area capacitance for a back gate
    For a metal back gate, set t_bgSiO2 = 0
    """
    return cst_epsilon0*1/(t_bg/cst_kappa_hBN + t_bgSiO2/cst_kappa_SiO2)

def calc_nD(V_tg, V_bg, c_tg, c_bg, n_offset=0, D_offset = 0):
    n = (1/cst_e) * (c_tg*V_tg  + c_bg*V_bg) + n_offset
    D = 0.5*(c_tg*V_tg - c_bg*V_bg)*1/cst_epsilon0 + D_offset
    return n, D


# QC0426 gate stack (per notebook index): SiO2 term zeroed — (n, D) uses the
# hBN top + back gates only; the Si gate (k_2410) is a fixed screening offset.
t_tg = 15.6e-9  # m (= 1.3 * t_bg, consistent with the CNP slope Vtg ≈ -1.3·Vbg)
t_bg = 12e-9    # m
t_bgSiO2 = 0    # m — zeroed for QC0426

c_tg = calc_ctg(t_tg)
c_bg = calc_cbg(t_bg, t_bgSiO2)

# Measurement config

In [15]:
# Active QC0426 config (device card / runs 79-84 wiring).
meas_config = {
    "sr0": "drain current on 42",  # drain
    "sr0_gain": -1e6,  # ithaco gain
    "sr1": "09-01", # Rxx pairs
    "sr2": "12-20", # Rxx pairs
    "sr3": "09-12", # Rxy pairs
    "bias_resistor": 100e6,
    "bias_voltage": sr1.amplitude(),
    "lockin_freq": sr1.frequency(),
    "sr0_sens": sr0.sensitivity(),
    "sr1_sens": sr1.sensitivity(),
    "sr2_sens": sr2.sensitivity(),
    "sr3_sens": sr3.sensitivity(),
    "sample_temperature:": heater.sample_temperature(),  # key verbatim (trailing ':') to match this cooldown's metadata
    "Si (V)": k_2410.volt(),
    "comments": "B-n sweep at D=+0.5, DC205 for TG, K2400 for BG, K2410 for Si gate, source current on 33, float other contacts",
}


## HERA — preflight probe (16:28)
RTC cell-append test on the measure notebook. If you can read this in the browser, the nb-collab path is live. No action needed — leave as a session breadcrumb.

## HERA — bring-up status + Gate B fix (first cells)

Gates A is green: setup ran in this kernel (magnet readback 0.002 T set-value parked, base T = 1.157 K, all four SR830s + both Keithleys + DC205 connected). Gate B is NOT green: the SAFETIES cell committed Si validator [-80, +100] V but the device card's contract is [0, +100] V. The cell below tightens Si to the card values — edit before running if you intend a different floor (that resolves the card's VERIFY on Si range).

Operator confirms needed before the first sweep (verbal is fine): (1) contact wiring unchanged since run 84 (sr1=09-01, sr2=12-20, sr3=09-12, drain on 42 — matches meas_config); (2) Si at +50 V for this session (D-series convention, matches the reference pack — assumed below); (3) FastBi1K5 set call blocks, cap 8 T; (4) helium/schedule good for ~5 h. Magnet-at-0 and base-T are confirmed from readbacks. Note: meas_config 'comments' field is stale ('B-n sweep at D=+0.5 ... source current on 33') — my cells override it per-run; if the source really moved from 42, stop me.

In [16]:
# HERA: SAFETIES amendment — Si validator to device-card range [0, +100] V.
# The SAFETIES cell above committed SI_MIN = -80 V; the device card's gate-B
# contract is [0, +100] V (deliberately narrower than the reference campaign).
SI_MIN, SI_MAX = 0.0, +100.0
k_2410.volt.vals = validators.Numbers(SI_MIN, SI_MAX)
print('validators committed:')
print(f'  TG [{TG_MIN}, {TG_MAX}] V   BG [{BG_MIN}, {BG_MAX}] V')
print(f'  Si [{SI_MIN}, {SI_MAX}] V   |B| <= {B_SESSION_CAP} T')

validators committed:
  TG [-6.5, 7.5] V   BG [-5.0, 4.8] V
  Si [0.0, 100.0] V   |B| <= 8.0 T


## HERA — Gate C baseline 1: k_bg response + leakage (~4 min)

Qualitative check (no designated reference trace): smoothness, leakage floor on k_bg current, CNP visibility. Precondition: TG and BG at 0 V. Ramps Si to +50 V first (2 V steps, ~10 s), then sweeps k_bg 0 -> -5.0 -> +4.8 -> 0 V, 199 pts at 1.2 s/pt. B = 0 throughout. Ends with BG parked at 0.

In [17]:
# HERA: gate-C baseline 1 — k_bg gate response + leakage at B = 0.
for v in np.linspace(k_2410.volt(), 50.0, 26):   # Si to session point +50 V
    k_2410.volt(v)
    time.sleep(0.2)

bg_vec = np.hstack((
    np.linspace(0.0, BG_MIN, 51),       # 0 -> -5.0
    np.linspace(BG_MIN, BG_MAX, 99),    # -5.0 -> +4.8
    np.linspace(BG_MAX, 0.0, 49),       # +4.8 -> 0 (park)
))

mc = dict(meas_config)
mc['Si (V)'] = k_2410.volt()
mc['comments'] = 'gate-C re-entry baseline 1: k_bg response/leakage, B=0, TG=0, Si=+50V'

s = async_station(mc)
s.add_comment('gate-current-sanity-check')
s.add_comment('k_bg 0 -> -5.0 -> +4.8 -> 0 V, 199 pts, delay 1.2 s, B=0, TG=0 V, Si=+50 V, T~1.16 K')
s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P)
s.fp(k_bg.curr)
s.sweep(k_bg.volt, bg_vec, delay=1.2)

2026-06-10 16:45:52,968 [INFO] Starting sweep with ID 0
2026-06-10 16:45:52,969 [INFO] Minimum duration 0h 3m 58s


2026-06-10 16:50:37,699 [INFO] Completed in 0h 4m 44s
2026-06-10 16:50:37,700 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\0\data.tsv.gz


## HERA — Gate C baseline 2: density cut, D = +0.25 V/nm (~4 min)

Exact reproduction of reference/baseline_B0_densitycut_Dp0p25.csv: vtg 1.0 -> 5.0 V (201 pts), vbg = vtg/1.3 - 2.0 (D-line const -2 -> D = +0.25), B = 0, Si = +50 V. I compare against the pack's four checks (CNP peak ~4.5 kOhm on sr1, CNP within 0.1e12 of n=0, shape, drain ~5 nA) as soon as it lands; disagreement is stop-and-report. Gates end at (vtg, vbg) = (5.0, +1.85) V.

In [18]:
# HERA: gate-C baseline 2 — B=0 density cut along D = +0.25 V/nm,
# reproducing reference/baseline_B0_densitycut_Dp0p25.csv.
vtg_vec = np.linspace(1.0, 5.0, 201)
vbg_vec = vtg_vec / 1.3 - 2.0          # const = -2 -> D = +0.25 V/nm
vtg_vec, vbg_vec = sanitize_tg_bg(vtg_vec, vbg_vec, tg_lims, bg_lims)

mc = dict(meas_config)
mc['Si (V)'] = k_2410.volt()
mc['comments'] = 'gate-C re-entry baseline 2: B=0 density cut, D=+0.25 V/nm, vs reference pack'

s = async_station(mc)
s.add_comment('trilayer-stacking-id')
s.add_comment('B=0 density cut D=+0.25: vtg 1.0->5.0 V (201 pts), vbg=vtg/1.3-2.0, delay 1.2 s, Si=+50 V, T~1.16 K')
s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P)
s.multisweep([dc205_volt, k_bg.volt], [vtg_vec, vbg_vec], delay=1.2)

2026-06-10 16:54:18,338 [INFO] Starting multisweep with ID 1
2026-06-10 16:54:18,339 [INFO] Minimum duration 0h 4m 1s


2026-06-10 16:58:54,782 [INFO] Completed in 0h 4m 36s
2026-06-10 16:58:54,782 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\1\data.tsv.gz


## HERA — recipe step 1: Rxx(CNP) vs D at B = 0 (~4 min)

Gate C passed (see ana notebook) — moving to the cheapest stacking discriminator (trilayer-stacking-id step 1). Single co-swept line along the fitted CNP locus Vtg = 0.06 - 1.3*Vbg, vbg -5.0 -> +4.8 V (197 pts, 1.2 s/pt): n stays ~0 (|n| < 0.07e12 end-to-end) while D runs +1.26 -> -1.19 V/nm, the full safe range. Readout: CNP Rxx growing orders of magnitude with |D| = ABC-leaning (gap opens); flat/weak/decreasing = ABA-leaning (band overlap). B = 0 throughout, Si = +50 V. Preconditions: nothing — gates ramp from wherever run 1 left them ((5.0, +1.85) V) to the line start (vtg +6.56, vbg -5.0). Gates end at (-6.18, +4.8) V.

In [19]:
# HERA: trilayer-stacking-id step 1 — CNP resistance vs displacement field, B=0.
vbg_vec = np.linspace(-5.0, 4.8, 197)
vtg_vec = 0.06 - 1.3 * vbg_vec        # fitted CNP locus (device card)
vtg_vec, vbg_vec = sanitize_tg_bg(vtg_vec, vbg_vec, tg_lims, bg_lims)

mc = dict(meas_config)
mc['Si (V)'] = k_2410.volt()
mc['comments'] = 'CNP-vs-D: co-swept CNP line, B=0, Si=+50 V, D +1.26 -> -1.19 V/nm'

s = async_station(mc)
s.add_comment('trilayer-stacking-id')
s.add_comment('step1 CNP-vs-D, B=0: vbg -5.0 -> +4.8 V (197 pts), vtg = 0.06 - 1.3*vbg, delay 1.2 s, Si=+50 V, T~1.16 K')
s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P)
s.multisweep([dc205_volt, k_bg.volt], [vtg_vec, vbg_vec], delay=1.2)

2026-06-10 17:04:33,196 [INFO] Starting multisweep with ID 2
2026-06-10 17:04:33,197 [INFO] Minimum duration 0h 3m 56s


2026-06-10 17:09:19,210 [INFO] Completed in 0h 4m 45s
2026-06-10 17:09:19,211 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\2\data.tsv.gz


## HERA — iteration-1 Landau fan: D = 0, electron side, 0 -> 8 T (~1 h 40 m)

Step-1 read (run 2, see chat/ana): CNP Rxx grows x150-180 with |D| on both Rxx pairs, two-terminal R reaching ~350 MOhm at D ~ +1 V/nm — strongly ABC-leaning. The D=0 fan is the independent discriminator: ABA shows the N_M=0 vertical crossing series (model predicts it at n ~ 0.5-1.2e12 for Datta parameters) and curved crossing chains; ABC shows one clean ladder, nu = +-6-centered.

Geometry: n-line -0.4 -> +4.0e12 cm-2 (121 pts, dn = 0.037e12) along D = 0 (vbg = vtg/1.3); B = 0 -> 8 T in 0.25 T rows (33 rows). Aliasing: dnu=4 structure resolved above ~0.75 T, integer rays above ~3 T — adequate for the hunt window; claimed features get the decimation test regardless. Wall-clock: ~1 h 40 m at the prior campaign's ~30 s/row overhead; if the magnet really ramps at 0.137 T/min it is ~2 h 25 m — the first rows calibrate which. Magnet is at 0 (no positioning ramp), ends at 8 T — please confirm the set call blocks per-row and current lock-in sensitivities are your fan settings before running. I re-form the partial fan from TSV as rows land and will flag early if it is washed out.

In [20]:
# HERA: trilayer-stacking-id step 3 — Landau fan, D = 0, electron side.
n_vec = np.linspace(-0.4e16, 4.0e16, 121)            # m^-2 = -0.4 -> +4.0 e12 cm^-2
vtg_vec = n_vec * cst_e / (2 * c_tg)                 # D = 0: vbg = vtg/1.3, n = 2*c_tg*vtg/e
vbg_vec = vtg_vec / 1.3
vtg_vec, vbg_vec = sanitize_tg_bg(vtg_vec, vbg_vec, tg_lims, bg_lims)

B_vec = np.arange(0.0, 8.0 + 1e-9, 0.25)             # 33 rows, 0 -> 8 T
assert np.abs(B_vec).max() <= B_SESSION_CAP

mc = dict(meas_config)
mc['Si (V)'] = k_2410.volt()
mc['comments'] = 'iteration-1 Landau fan: D=0 electron side, B 0->8 T, Si=+50 V'

s = async_station(mc)
s.add_comment('landau-fan')
s.add_comment('trilayer-stacking-id step 3: D=0, n -0.4->+4.0e12 (121 pts, vbg=vtg/1.3), B 0->8 T x 0.25 T (33 rows), fast_delay 1.2 s, Si=+50 V, T~1.16 K')
s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P)
s.multimegasweep([magnet.mag], [B_vec], [dc205_volt, k_bg.volt], [vtg_vec, vbg_vec],
                 slow_delay=5, fast_delay=1.2)

2026-06-10 17:23:38,788 [INFO] Starting multimegasweep with ID 3
2026-06-10 17:23:38,789 [INFO] Minimum duration 1h 22m 36s


2026-06-10 19:58:50,116 [INFO] Completed in 2h 35m 11s
2026-06-10 19:58:50,117 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\3\data.tsv.gz


## HERA — densified high-B segment: B 8 -> 6 T, fine n-grid (~55 min)

Run-3 verdicts (full analysis in ana): rays onset ~3.5-4 T; main Dnu~4 family through origin; a SECOND straight ray family offset by +0.45e12 (decimation-robust — real, the superposed-fan signature); no ABA crossing features where resolvable; Hall still classical (no plateaus). Remaining ambiguity is the absolute nu assignment (global density rescale 0.86 vs 1.09 both make integers). This segment resolves individual integer rays: n 0.3 -> 3.2e12 at 201 pts (dn = 0.0145e12; integer-ray spacing at 6 T = 0.145e12 = 10 pts/period — safely Nyquist-clear), B 8 -> 6 T descending in 0.25 T rows (9 rows; magnet is AT 8 T — no positioning ramp, no re-traverse). Covers the nu~6 edge candidate (n ~ 0.9-1.2), the offset family (n ~ 2-3), and the main family's low edge. ~356 s/row -> ~55 min total. Decides: nu=+-6-centered 12-fold sequence (ABC) vs other; offset-family integer spacing; plateau development at the nu~6 edge.

In [21]:
# HERA: trilayer-stacking-id step 4 — densified segment, B 8->6 T, D = 0.
n_vec = np.linspace(0.3e16, 3.2e16, 201)             # m^-2 = 0.3 -> 3.2 e12 cm^-2
vtg_vec = n_vec * cst_e / (2 * c_tg)
vbg_vec = vtg_vec / 1.3
vtg_vec, vbg_vec = sanitize_tg_bg(vtg_vec, vbg_vec, tg_lims, bg_lims)

B_vec = np.arange(8.0, 6.0 - 1e-9, -0.25)            # 9 rows, descending from current field
assert np.abs(B_vec).max() <= B_SESSION_CAP

mc = dict(meas_config)
mc['Si (V)'] = k_2410.volt()
mc['comments'] = 'densified fan segment: D=0, n 0.3->3.2e12 fine grid, B 8->6 T descending'

s = async_station(mc)
s.add_comment('landau-fan')
s.add_comment('trilayer-stacking-id step 4 densify: D=0, n 0.3->3.2e12 (201 pts, dn=0.0145e12), B 8->6 T x 0.25 (9 rows, descending), fast_delay 1.2 s, Si=+50 V, T~1.16 K')
s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P)
s.multimegasweep([magnet.mag], [B_vec], [dc205_volt, k_bg.volt], [vtg_vec, vbg_vec],
                 slow_delay=5, fast_delay=1.2)

2026-06-10 20:30:07,869 [INFO] Starting multimegasweep with ID 4
2026-06-10 20:30:07,870 [INFO] Minimum duration 0h 36m 55s


2026-06-10 21:27:08,941 [INFO] Completed in 0h 57m 1s
2026-06-10 21:27:08,941 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\4\data.tsv.gz


## HERA — overnight fine-B fan: D = 0, 6 -> 0 T, 0.05 T rows (~12.3 h)

Strengthens the stacking call at the discriminating D: (1) full coverage of the model's N_M=1-4 chain loci in the resolvable regime (n to 8e12, which run 3 lacked) at 5x B-sampling — a chain manifests across ~40-80 fine rows; (2) series-test at n~0.9-1.2 (N_M=0 / E3 anomaly band); (3) continuous nu=6-edge tracking. Finite-D caveat does not apply — this is D=0. Grid: n -0.5 -> +8.0e12 (271 pts, dn=0.0315; integer rays clear above ~2.6 T, dnu=4 above ~0.7 T), B descending 6 -> 0 in 121 rows; ends with the magnet AT 0 T (end-of-session parking included). Estimate 12.3 h (~345 s/row + 44 min total ramp), landing ~10:40 if started ~22:20. Unattended-run notes: every row steps toward zero field, so any failure mode is benign; partial rows salvageable; do NOT interrupt to stop the magnet — it does not stop the ramp.

In [ ]:
# HERA: trilayer-stacking-id — overnight fine-B fan, D = 0, 6 -> 0 T.
n_vec = np.linspace(-0.5e16, 8.0e16, 271)            # m^-2 = -0.5 -> +8.0 e12 cm^-2
vtg_vec = n_vec * cst_e / (2 * c_tg)                 # D = 0 line: vbg = vtg/1.3
vbg_vec = vtg_vec / 1.3
vtg_vec, vbg_vec = sanitize_tg_bg(vtg_vec, vbg_vec, tg_lims, bg_lims)

B_vec = np.arange(6.0, 0.0 - 1e-9, -0.05)            # 121 rows, descending to 0 T
assert np.abs(B_vec).max() <= B_SESSION_CAP

mc = dict(meas_config)
mc['Si (V)'] = k_2410.volt()
mc['comments'] = 'overnight fine-B fan: D=0, n -0.5->+8e12, B 6->0 T x 0.05 T, ends parked at 0 T'

s = async_station(mc)
s.add_comment('landau-fan')
s.add_comment('trilayer-stacking-id overnight: D=0, n -0.5->+8.0e12 (271 pts, dn=0.0315e12), B 6->0 T x 0.05 (121 rows, descending), fast_delay 1.2 s, slow_delay 5 s, Si=+50 V, T~1.16 K')
s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P)
s.multimegasweep([magnet.mag], [B_vec], [dc205_volt, k_bg.volt], [vtg_vec, vbg_vec],
                 slow_delay=5, fast_delay=1.2)

2026-06-10 22:00:21,315 [INFO] Starting multimegasweep with ID 5
2026-06-10 22:00:21,316 [INFO] Minimum duration 11h 5m 54s


IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



## HERA — LK mass series (1/3): base-T field sweep at the fixed point (~50 min)

Temperature series to split the two SdH frequencies by effective mass (ana idx 19: fork (i) ABA two-pocket vs (ii) parallel region). Fixed point: n_cap = 7.2e12, D = 0 -> (vtg, vbg) = (3.387, 2.605) V — the FFT sweet spot where f_low, f_main and sidebands are all strong. This cell: gentle gate move from the run-5 endpoint, then B 3 -> 6 T, 151 pts (0.02 T steps), heaters untouched (T ~ 1.2-1.3 K). First setpoint includes the 0 -> 3 T positioning ramp (~22 min), traverse ~22 min, total ~50 min. Temperature readbacks are fp'd (recipe: the ladder is read from data, never from setpoints).

In [23]:
# HERA: temperature-dependence (Flavor B) — base-T anchor sweep.
for v in np.linspace(dc205_volt(), 3.387, 10):
    dc205_volt(v); time.sleep(0.5)
for v in np.linspace(k_bg.volt(), 2.605, 10):
    k_bg.volt(v); time.sleep(0.2)
print(f'gates at vtg={dc205_volt():.3f}, vbg={k_bg.volt():.3f}, T={heater.sample_temperature():.3f} K')

B_vec = np.linspace(3.0, 6.0, 151)
assert np.abs(B_vec).max() <= B_SESSION_CAP

mc = dict(meas_config)
mc['Si (V)'] = k_2410.volt()
mc['comments'] = 'LK series base point: B 3->6 T at n_cap=7.2e12, D=0, heaters off, T~1.25 K'

s = async_station(mc)
s.add_comment('temperature-dependence')
s.add_comment('LK ladder pt 1/6: T=base (heaters off), B 3->6 T (151 pts, 0.02 T), n_cap=7.2e12 (vtg 3.387, vbg 2.605), D=0, Si=+50 V')
s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P)
s.fp(heater.sample_temperature).fp(heater.vaporizer_temperature)
s.sweep(magnet.mag, B_vec, delay=1.2)

gates at vtg=3.387, vbg=2.605, T=1.159 K
2026-06-11 13:45:05,295 [INFO] Starting sweep with ID 6
2026-06-11 13:45:05,296 [INFO] Minimum duration 0h 3m 1s


2026-06-11 14:39:31,838 [INFO] Completed in 0h 54m 26s
2026-06-11 14:39:31,839 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\6\data.tsv.gz


## HERA — measured ramp to 0 T (~50 min, replaces a bare ramp)

Per Bingcheng: heat-up must happen at 0 T, with a manual needle-valve precondition. This cell does the required 6 -> 0 T ramp as a recorded sweep at the LK fixed point (gates untouched): the 6 -> 3 segment is a direction-reversed reproducibility check on run 6 (recipe Verify), and 3 -> 0 extends the base-T trace for free. Same wall-clock as an unrecorded ramp (~44 min ramp + ~6 min point delays). Ends: B = 0, ready for the needle-valve step.

In [24]:
# HERA: ramp 6 -> 0 T as a measured base-T sweep (LK fixed point, heaters off).
B_vec = np.linspace(6.0, 0.0, 301)
assert np.abs(B_vec).max() <= B_SESSION_CAP

mc = dict(meas_config)
mc['Si (V)'] = k_2410.volt()
mc['comments'] = 'measured ramp-down 6->0 T at LK fixed point (n_cap=7.2e12, D=0), heaters off; 6->3 = run-6 reproducibility check'

s = async_station(mc)
s.add_comment('temperature-dependence')
s.add_comment('LK anchor repeat + descent: B 6->0 T (301 pts), n_cap=7.2e12, D=0, Si=+50 V, heaters off, T~1.2 K')
s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P)
s.fp(heater.sample_temperature).fp(heater.vaporizer_temperature)
s.sweep(magnet.mag, B_vec, delay=1.2)

2026-06-11 14:58:21,478 [INFO] Starting sweep with ID 7
2026-06-11 14:58:21,479 [INFO] Minimum duration 0h 6m 1s


VisaIOError: ('VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.', "asking 'KRDG? A' to <Lakeshore_335: l335_3>", 'getting l335_3_sample_temperature')

In [32]:
magnet.mag(0)

## Bingcheng and Zack - Warmup procedure

Requires some manual steps to prepare for heating to 25K

- needs needle valve actuation before the heating begins
- heating needs to happen at 0T

In [37]:
heater.sample_range(1)
heater.vaporizer_range(1)
heater.sample_power(70)
heater.vaporizer_power(70)

In [127]:
print(heater.sample_temperature(), heater.vaporizer_temperature())

18.491 15.325


In [179]:
heater.sample_range(1)
heater.vaporizer_range(1)
heater.sample_power(10)
heater.vaporizer_power(10)

In [199]:
print(heater.sample_temperature(), heater.vaporizer_temperature())

18.776 17.148


In [200]:
# run this cell after both sample & vaporizor temperature exceed 4K 
heater.set_sample_closed_loop()
heater.set_vaporizer_closed_loop()
heater.set_both(20, ratio = 0.99) # set to 20K since it's already ~17-18K and HERA wants to go to 25K
# open needle valve to ~10 o'clock position (currently at ~3 o'clock, so almost 1/2 of a turn)

In [201]:
# adjusted the needle valve

## HERA — LK series (2/3 v2): heater bring-up to 25 K at 0 T (supervised)

PRECONDITIONS (hard): (1) magnet at 0 T (cell above completed); (2) Bingcheng's manual needle-valve step DONE. Then this cell sets both setpoints to 25 K and prints readbacks every 30 s (up to 15 min). If T is flat with 0% heater power, the output RANGE is not enabled — set heater.sample_range / vaporizer_range to the values Bingcheng prescribes and re-run. Edit the set_both ratio if his procedure says so.

In [203]:
# HERA: heater bring-up at 0 T — run only after needle-valve precondition.
# assert abs(magnet.mag()) < 0.01, 'magnet not at 0 T' # incorrect syntax, no magnet.mag().  confirmed manually field is 0
heater.set_both(25.0, ratio = 0.99)
# heater.sample_range(<n>); heater.vaporizer_range(<n>)   # <-- per Bingcheng
print('setpoints:', heater.sample_setpoint(), heater.vaporizer_setpoint())
for i in range(30):
    ts = heater.sample_temperature(); tv = heater.vaporizer_temperature()
    pw = heater.sample_power()
    print(f'{i*30:4d} s: sample {ts:6.3f} K  vaporizer {tv:6.3f} K  htr1 {pw:5.1f}%')
    if abs(ts - 25.0) < 0.15:
        print('at setpoint.'); break
    time.sleep(30)

setpoints: 20.0 19.8
   0 s: sample 20.216 K  vaporizer 19.799 K  htr1   0.4%
  30 s: sample 25.116 K  vaporizer 24.768 K  htr1   3.2%
at setpoint.


## HERA — LK series (3/3 final): descending T ladder 25 -> 5 K (~3.5 h, Aaron-approved)

T-steps at field approved (Aaron); heat-up still at 0 T per Bingcheng (bring-up cell). PRECONDITION: **Grafana Lakeshore logging PAUSED for the whole heated period** — run 7 died from logger/VISA contention; the preflight below hard-fails if it is still up. Five points 25/20/15/10/5 K descending, settle-on-readback (|T-sp| < 0.1 K, drift < 50 mK/30 s, 20 min cap), identical 151-pt sweep 3 <-> 6 T alternating direction (first sweep 3 -> 6; its first setpoint includes the 0 -> 3 T positioning ramp, ~22 min). One file per T. Ends: heaters OFF, magnet at 6 T... ends at 6 T after the 5th sweep; recool to base from there. LK recap: heavy branch (m* ~ 0.05) suppressed 4-8x by 20-25 K, light (~0.025) only ~2x; equal damping = same-mass = region scenario. Run after the bring-up cell is stable at 25 K.

In [207]:
magnet.heater_on()

In [210]:
# HERA: LK ladder (final) — descending 25 -> 5 K, alternating 3<->6 T sweeps.
# Preflight: Lakeshore read latency (catches a still-running Grafana logger).
for _ in range(5):
    t0 = time.time(); _ = heater.sample_temperature(); dt = time.time() - t0
    assert dt < 2.0, f'Lakeshore read took {dt:.1f} s — logger still running? PAUSE IT.'
print('Lakeshore preflight OK')

ladder = [25.0, 20.0, 15.0, 10.0, 5.0]
up = np.linspace(3.0, 6.0, 151)
assert np.abs(up).max() <= B_SESSION_CAP

mc = dict(meas_config)
mc['Si (V)'] = k_2410.volt()
mc['comments'] = 'LK ladder: T 25->5 K descending, B 3<->6 T alternating, n_cap=7.2e12, D=0'

s = async_station(mc)
s.add_comment('temperature-dependence')
s.add_comment('LK ladder pts 2-6/6: T = 25,20,15,10,5 K, B 3<->6 T alternating (151 pts), n_cap=7.2e12, D=0, Si=+50 V')
s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P)
s.fp(heater.sample_temperature).fp(heater.vaporizer_temperature)

for i, Tsp in enumerate(ladder):
    heater.set_both(Tsp, ratio = 0.99) # ratio added by Zack to see if it fixes the T tolerance issue
    t0 = time.time(); prev = heater.sample_temperature()
    # while time.time() - t0 < 20*60:
    #     time.sleep(30)
    #     ts = heater.sample_temperature()
    #     tv = heater.vaporizer_temperature()
    #     if abs(ts - Tsp) < 0.10 and abs(ts - prev) < 0.05:
    #         break
    #     print(f'Target T {Tsp};  Current T_sample {ts};  Current T_vaporizer: {tv}; Duration {time.time() - t0:.1f}s') # added by Zack for debugging, found that T = 25.6 about constant causing it to loop indefinitely
    #     prev = ts
    
    # settle on stability only — T is recorded; LK uses measured T, not setpoint
    prev = heater.sample_temperature(); stable = 0
    while time.time() - t0 < 20*60:
      time.sleep(30)
      ts = heater.sample_temperature()
      if abs(ts - prev) < 0.025:      # < ~50 mK/min
          stable += 1
          if stable >= 2: break       # two consecutive stable intervals
      else:
          stable = 0
      prev = ts
    print(f'T {Tsp} K: settled at {heater.sample_temperature():.3f} K, '
          f'{(time.time()-t0)/60:.1f} min')
    vec = up if i % 2 == 0 else up[::-1]   # first sweep 3->6 (magnet arrives from 0)
    s.sweep(magnet.mag, vec, delay=1.2)

heater.off()
print('ladder done: heaters OFF, magnet at 6 T, recooling to base')

Lakeshore preflight OK
T 25.0 K: settled at 26.193 K, 1.0 min
2026-06-11 19:45:08,129 [INFO] Starting sweep with ID 8
2026-06-11 19:45:08,131 [INFO] Minimum duration 0h 3m 1s


2026-06-11 20:39:35,148 [INFO] Completed in 0h 54m 26s
2026-06-11 20:39:35,149 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\8\data.tsv.gz
T 20.0 K: settled at 21.968 K, 20.0 min
2026-06-11 20:59:35,395 [INFO] Starting sweep with ID 9
2026-06-11 20:59:35,397 [INFO] Minimum duration 0h 3m 1s


2026-06-11 21:32:11,913 [INFO] Completed in 0h 32m 36s
2026-06-11 21:32:11,914 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\9\data.tsv.gz
T 15.0 K: settled at 16.172 K, 20.0 min
2026-06-11 21:52:12,166 [INFO] Starting sweep with ID 10
2026-06-11 21:52:12,169 [INFO] Minimum duration 0h 3m 1s


2026-06-11 22:24:48,136 [INFO] Completed in 0h 32m 35s
2026-06-11 22:24:48,136 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\10\data.tsv.gz
T 10.0 K: settled at 10.002 K, 4.5 min
2026-06-11 22:29:18,212 [INFO] Starting sweep with ID 11
2026-06-11 22:29:18,214 [INFO] Minimum duration 0h 3m 1s


2026-06-11 23:01:54,484 [INFO] Completed in 0h 32m 36s
2026-06-11 23:01:54,485 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\11\data.tsv.gz
T 5.0 K: settled at 5.022 K, 2.0 min
2026-06-11 23:03:54,523 [INFO] Starting sweep with ID 12
2026-06-11 23:03:54,524 [INFO] Minimum duration 0h 3m 1s


2026-06-11 23:36:31,038 [INFO] Completed in 0h 32m 36s
2026-06-11 23:36:31,039 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\12\data.tsv.gz
ladder done: heaters OFF, magnet at 6 T, recooling to base


In [212]:
# HERA: cooldown to base — heaters off, block until cold, halt queue on timeout.
heater.off()
T_TARGET = 1.5      # K — cold enough for all frequency-domain work (base ~1.2)
t0 = time.time()
while True:
  ts = heater.sample_temperature(); tv = heater.vaporizer_temperature()
  if ts < T_TARGET:
      print(f'cooled: sample {ts:.3f} K, vaporizer {tv:.3f} K '
            f'after {(time.time()-t0)/60:.1f} min')
      break
  if time.time() - t0 > 2*3600:
      raise RuntimeError(f'recool timeout: T_sample {ts:.3f} K after 2 h — investigate')
  print(f'{(time.time()-t0)/60:5.1f} min: sample {ts:6.3f} K  vaporizer {tv:6.3f} K')
  time.sleep(60)

cooled: sample 1.317 K, vaporizer 0.000 K after 0.0 min


In [214]:
magnet.get_field()

6.002123068493151

## HERA — deciding measurement: f_low vs D at fixed n (~2 h, run after recool)

Three field sweeps at n_cap = 7.2e12, D = +0.2 / +0.4 / -0.2 V/nm (D = 0 reference = runs 5/6/7 + ladder base anchors). B 2 <-> 6 T (201 pts, alternating direction starting downward from the ladder's 6 T endpoint; wider window -> 3 T FFT bins on f_low). Readout (model-corrected, see chat/ana): ABA pockets are nearly D-independent (predicted f_ML shift < ~10%) — f_low STAYING PUT favors ABA two-pocket; the region-beat scenario requires a different-lever-arm region whose density shifts by ~1e12 under the gate rebalancing — f_low MOVING by tens of T favors the second-region explanation. Run at base T after the ladder recool (frequencies are T-independent but amplitude comparability is cleanest cold). Grafana can stay paused or be re-enabled and T-following dropped for these — frequencies do not need the T column; cell keeps fp(T) and assumes the logger is still paused.

In [215]:
# HERA: f_low vs D — three sweeps at fixed n_cap = 7.2e12, base T.

# --- preconditions: queued unattended, halt on any failure ---
ts = heater.sample_temperature()
assert ts < 1.5, f'not recooled: T_sample = {ts:.2f} K'
assert heater.sample_power() < 0.5, f'sample heater still active ({heater.sample_power():.1f}%)'
b = magnet.get_field()
assert 5.7 <= abs(b) <= 6.3, f'magnet at {b:.2f} T, expected ~6 T (ladder endpoint)'
t0 = time.time(); _ = heater.sample_temperature()
assert time.time() - t0 < 2.0, 'Lakeshore read slow — Grafana logger running again?'
print(f'preconditions OK: T={ts:.3f} K, B={b:.2f} T')

# vtg = (n*e/2 + eps0*D)/c_tg ; vbg = (n*e/2 - eps0*D)/c_bg  (D in V/m)
pts = [(+0.2e9, 4.427, 1.805),
       (+0.4e9, 5.467, 1.005),
       (-0.2e9, 2.347, 3.405)]
up = np.linspace(2.0, 6.0, 201)
assert np.abs(up).max() <= B_SESSION_CAP

mc = dict(meas_config)
mc['Si (V)'] = k_2410.volt()
mc['comments'] = 'f_low vs D: n_cap=7.2e12 fixed, D=+0.2/+0.4/-0.2 V/nm, B 2<->6 T, base T'

s = async_station(mc)
s.add_comment('temperature-dependence')
s.add_comment('f_low-vs-D decider: n_cap=7.2e12, D=+0.2,+0.4,-0.2 V/nm, B 2<->6 T (201 pts) alternating, base T, Si=+50 V')
s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P)
s.fp(heater.sample_temperature).fp(heater.vaporizer_temperature)

for i, (D, vtg_t, vbg_t) in enumerate(pts):
    for v in np.linspace(dc205_volt(), vtg_t, 12):
        dc205_volt(v); time.sleep(0.5)
    for v in np.linspace(k_bg.volt(), vbg_t, 12):
        k_bg.volt(v); time.sleep(0.2)
    print(f'D = {D/1e9:+.2f} V/nm: gates ({dc205_volt():.3f}, {k_bg.volt():.3f}), '
          f'T = {heater.sample_temperature():.3f} K')
    vec = up[::-1] if i % 2 == 0 else up   # first sweep 6->2 (from ladder endpoint)
    s.sweep(magnet.mag, vec, delay=1.2)

print('f_low-vs-D set done; magnet at 2 T, gates at D=-0.2 point')

preconditions OK: T=1.317 K, B=6.00 T
D = +0.20 V/nm: gates (4.427, 1.805), T = 1.317 K
2026-06-12 00:12:13,347 [INFO] Starting sweep with ID 13
2026-06-12 00:12:13,349 [INFO] Minimum duration 0h 4m 1s


2026-06-12 00:55:40,500 [INFO] Completed in 0h 43m 27s
2026-06-12 00:55:40,501 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\13\data.tsv.gz
D = +0.40 V/nm: gates (5.467, 1.005), T = 1.262 K
2026-06-12 00:55:51,190 [INFO] Starting sweep with ID 14
2026-06-12 00:55:51,191 [INFO] Minimum duration 0h 4m 1s


2026-06-12 01:39:18,212 [INFO] Completed in 0h 43m 26s
2026-06-12 01:39:18,212 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\14\data.tsv.gz
D = -0.20 V/nm: gates (2.347, 3.405), T = 1.308 K
2026-06-12 01:39:31,964 [INFO] Starting sweep with ID 15
2026-06-12 01:39:31,966 [INFO] Minimum duration 0h 4m 1s


2026-06-12 02:22:59,229 [INFO] Completed in 0h 43m 27s
2026-06-12 02:22:59,230 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\15\data.tsv.gz
f_low-vs-D set done; magnet at 2 T, gates at D=-0.2 point


## HERA — SAFETIES amendment: field cap to approved 14 T

Operator approval 06-11 (Qingrui, with Aaron): |B| <= 14 T is the new session cap (cap, not endpoint). Run before the E3 segment below (which goes to 9 T). Gate validators unchanged.

In [216]:
B_SESSION_CAP = 13.5  # T — operator-approved 06-11 (was 8.0) # modified by Zack for extra safety, headroom not needed
print(f'B_SESSION_CAP = {B_SESSION_CAP} T; gate validators unchanged:')
print(f'  TG [{TG_MIN}, {TG_MAX}] V   BG [{BG_MIN}, {BG_MAX}] V   Si [{SI_MIN}, {SI_MAX}] V')

B_SESSION_CAP = 13.5 T; gate validators unchanged:
  TG [-6.5, 7.5] V   BG [-5.0, 4.8] V   Si [0.0, 100.0] V


## HERA — E3 anomaly segment: 9 -> 6.5 T fine map around n ~ 1.0 (~2.5 h incl. ramp)

The last unexplained run-4/5 feature: a non-dispersing Rxx minimum at n ~ 1.02e12 seen at 7.25-8 T only (ana idx 17, E3). This segment decides series-vs-accident: B 9 -> 6.5 T in 0.05 T rows (51), n 0.7 -> 1.4e12 (71 pts, dn = 0.01) at D = 0. If it is a vertical bead/dark series (two-ladder physics) more crossings should appear toward 9 T at ~fixed n; if isolated, the window above 8 T is empty. First setpoint includes the 2 -> 9 T positioning ramp (~51 min). Requires the 14 T cap cell above. Queue after the f_low-vs-D cell; ends at 6.5 T.

In [217]:
# HERA: E3 anomaly — fine (n, B) segment 9 -> 6.5 T, D = 0.

# --- preconditions ---
ts = heater.sample_temperature()
assert ts < 1.5, f'not at base: T_sample = {ts:.2f} K'
assert B_SESSION_CAP >= 9.0, 'field-cap re-commit cell not run (need >= 9 T)'
b = magnet.get_field()
assert 1.7 <= abs(b) <= 2.3, f'magnet at {b:.2f} T, expected ~2 T (D-set endpoint) — did the D-set complete?'
print(f'preconditions OK: T={ts:.3f} K, B={b:.2f} T, cap={B_SESSION_CAP} T')

n_vec = np.linspace(0.7e16, 1.4e16, 71)
vtg_vec = n_vec * cst_e / (2 * c_tg)
vbg_vec = vtg_vec / 1.3
vtg_vec, vbg_vec = sanitize_tg_bg(vtg_vec, vbg_vec, tg_lims, bg_lims)

B_vec = np.arange(9.0, 6.5 - 1e-9, -0.05)   # 51 rows descending
assert np.abs(B_vec).max() <= B_SESSION_CAP

mc = dict(meas_config)
mc['Si (V)'] = k_2410.volt()
mc['comments'] = 'E3 anomaly segment: D=0, n 0.7-1.4e12 fine, B 9->6.5 T x 0.05; tests vertical-series hypothesis'

s = async_station(mc)
s.add_comment('trilayer-stacking-id')
s.add_comment('E3 test: D=0, n 0.7->1.4e12 (71 pts, dn=0.01), B 9->6.5 T x 0.05 (51 rows), fast_delay 1.2 s, Si=+50 V, base T')
s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P)
s.fp(heater.sample_temperature).fp(heater.vaporizer_temperature)
s.multimegasweep([magnet.mag], [B_vec], [dc205_volt, k_bg.volt], [vtg_vec, vbg_vec],
                 slow_delay=5, fast_delay=1.2)

preconditions OK: T=1.255 K, B=2.00 T, cap=13.5 T
2026-06-12 02:22:59,319 [INFO] Starting multimegasweep with ID 16
2026-06-12 02:22:59,319 [INFO] Minimum duration 1h 16m 40s


2026-06-12 05:01:07,571 [INFO] Completed in 2h 38m 8s
2026-06-12 05:01:07,572 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260610-QC0426-hera-live\data\16\data.tsv.gz


## HERA — end-of-night ramp to 0 T (~47 min)

Bare (unmeasured) ramp per operator request; queue last. Ends the night: magnet 0 T, heaters off (since the ladder), base T — clean start state for tomorrow.

In [218]:
# HERA: park magnet at 0 T.
magnet.mag(0.0)
print('magnet at 0 T:')
magnet.status()

magnet at 0 T:
---------- Magnet Status ----------
Field:               : -0.001 Tesla
Current:             : -0.011 A
Voltage:             : -0.372 V
Output Enabled       : True
Persistent Switch On : True
Loop Mode            : CC
Update Mode          : NORMAL
-----------------------------------
